# UTTA-Med — FIX: save-best source + Tent/EATA lr on OOD val

Settings: GPU on, Internet on.

If you Add Input the **Colab** `camelyon17_resnet18_source_s42.pt` (val AUROC ~0.9817), training is skipped.
Do **not** use the Kaggle epoch-8 overwrite checkpoint (ood 0.9431).

In [ ]:
!pip -q install datasets pyarrow scikit-learn tqdm pillow
import torch; assert torch.cuda.is_available(), 'Enable GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
import json, random, time
from pathlib import Path
import numpy as np, torch, torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm
from datasets import load_dataset

SEED, BATCH, LR_SRC, EPOCHS, PATIENCE = 42, 128, 1e-4, 20, 5
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda')

def find_good_ckpt():
    cands = list(Path('/kaggle/input').rglob('*.pt')) + list(WORK.glob('*.pt'))
    for p in cands:
        try:
            st = torch.load(p, map_location='cpu', weights_only=False)
            va = float(st.get('val_auroc', -1)); ep = int(st.get('epoch', -1))
            print('found', p, 'epoch', ep, 'val_auroc', va)
            # reject the known-bad last-epoch overwrite (epoch 8, labeled 0.979 but was not restored)
            if va >= 0.978 and ep <= 5:
                return p, st
            if va >= 0.980:
                return p, st
        except Exception as e:
            print('skip', p, e)
    return None, None

good_path, good_st = find_good_ckpt()
print('usable pretrained ckpt', good_path)

In [ ]:
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)
train_tf = transforms.Compose([transforms.Resize((96,96)), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10), transforms.ColorJitter(0.1,0.1,0.1,0.02), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
eval_tf = transforms.Compose([transforms.Resize((96,96)), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
def cid(v): return int(v.item() if hasattr(v,'item') else v)
class HFCamelyon(Dataset):
    def __init__(self, split, centers, tf):
        self.tf=tf; centers={int(c) for c in centers}
        keep=[i for i,c in enumerate(split['center']) if cid(c) in centers]
        self.ds = split.select(keep) if len(keep)<len(split) else split
        print(' kept', len(self.ds), sorted(centers))
    def __len__(self): return len(self.ds)
    def __getitem__(self,i):
        r=self.ds[i]; img=r['image']
        img = img.convert('RGB') if isinstance(img, Image.Image) else Image.fromarray(np.array(img)).convert('RGB')
        return self.tf(img), torch.tensor(float(int(r['label'])))
class Net(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__(); w=models.ResNet18_Weights.DEFAULT if pretrained else None
        bb=models.resnet18(weights=w); d=bb.fc.in_features; bb.fc=nn.Identity(); self.backbone=bb
        self.classifier=nn.Sequential(nn.Linear(d,d), nn.ReLU(True), nn.Dropout(0.5), nn.Linear(d,1))
    def forward(self,x): return self.classifier(self.backbone(x))

raw=load_dataset('wltjr1007/Camelyon17-WILDS')
train_set,id_set,ood_set,test_set = (HFCamelyon(raw['train'],{0,3,4},train_tf),
    HFCamelyon(raw['validation'],{0,3,4},eval_tf), HFCamelyon(raw['validation'],{1},eval_tf), HFCamelyon(raw['test'],{2},eval_tf))
def LD(ds,sh,bs=BATCH): return DataLoader(ds,batch_size=bs,shuffle=sh,num_workers=0,pin_memory=True)
train_loader,id_loader,ood_loader,test_loader = LD(train_set,True),LD(id_set,False),LD(ood_set,False),LD(test_set,False,64)
print('num_workers=0 to avoid Kaggle worker warnings')

In [ ]:
def ece_score(y,p,n=15):
    y,p=np.asarray(y).ravel(),np.asarray(p).ravel(); bins=np.linspace(0,1,n+1); e=0.0
    for i in range(n):
        m=(p>bins[i])&(p<=bins[i+1]) if i else (p>=bins[i])&(p<=bins[i+1])
        if m.any(): e += m.mean()*abs(y[m].mean()-p[m].mean())
    return float(e)
def metrics_of(y,p):
    y,p=np.asarray(y).ravel(),np.asarray(p).ravel(); pred=(p>=0.5).astype(int)
    return dict(n=int(len(y)), accuracy=float(accuracy_score(y,pred)),
        precision=float(precision_score(y,pred,zero_division=0)), recall=float(recall_score(y,pred,zero_division=0)),
        sensitivity=float(recall_score(y,pred,zero_division=0)),
        specificity=float(((y==0)&(pred==0)).sum()/max((y==0).sum(),1)),
        f1=float(f1_score(y,pred,zero_division=0)), auroc=float(roc_auc_score(y,p)),
        auprc=float(average_precision_score(y,p)), ece=ece_score(y,p), brier=float(np.mean((p-y)**2)))
def flip_table(y,b,a):
    y=np.asarray(y).ravel().astype(int); b=(np.asarray(b).ravel()>=0.5).astype(int); a=(np.asarray(a).ravel()>=0.5).astype(int)
    return dict(correction_rate=float(((b!=y)&(a==y)).sum()/max((b!=y).sum(),1)),
                harm_rate=float(((b==y)&(a!=y)).sum()/max((b==y).sum(),1)), flip_rate=float((a!=b).mean()),
                correction=int(((b!=y)&(a==y)).sum()), harmful=int(((b==y)&(a!=y)).sum()))
@torch.no_grad()
def evaluate(m,ld):
    m.eval(); ys,ps=[],[]
    for x,y in ld:
        x=x.to(device,non_blocking=True); ps.append(torch.sigmoid(m(x)).view(-1).cpu().numpy()); ys.append(y.view(-1).numpy())
    return metrics_of(np.concatenate(ys), np.concatenate(ps))

CKPT = WORK/'camelyon17_resnet18_source_s42_BEST.pt'
history=[]
if good_path is not None:
    print('Skip train; copying good ckpt'); st=good_st
    torch.save(st, CKPT)
else:
    print('Retrain with SAVE-BEST-ONLY and restore')
    model=Net(pretrained=True).to(device)
    opt=torch.optim.Adam(model.parameters(), lr=LR_SRC, weight_decay=1e-4)
    crit=nn.BCEWithLogitsLoss(); scaler=torch.amp.GradScaler('cuda')
    best,stale,best_state=-1.0,0,None
    for epoch in range(1,EPOCHS+1):
        model.train(); run=n=0
        for x,y in tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}'):
            x,y=x.to(device), y.view(-1,1).to(device); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'): loss=crit(model(x),y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run+=loss.item()*x.size(0); n+=x.size(0)
        val=evaluate(model,ood_loader)
        history.append(dict(epoch=epoch,train_loss=run/max(n,1),**{f'val_{k}':v for k,v in val.items()}))
        print(f"epoch {epoch:02d} loss={run/max(n,1):.4f} val_auroc={val['auroc']:.4f} f1={val['f1']:.4f} ece={val['ece']:.4f}")
        if val['auroc']>best:
            best,stale=val['auroc'],0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            torch.save({'model':best_state,'epoch':epoch,'val_auroc':best,'seed':SEED}, CKPT)
            print('  NEW BEST saved epoch', epoch)
        else:
            stale+=1
            print('  no improve stale', stale)
            if stale>=PATIENCE: print('early stop'); break
    assert best_state is not None
    model.load_state_dict(best_state)
    st=torch.load(CKPT,map_location='cpu',weights_only=False)

def load_fresh():
    m=Net(pretrained=False).to(device); m.load_state_dict(torch.load(CKPT,map_location='cpu',weights_only=False)['model']); return m
model=load_fresh()
id_m,ood_m,test_m=evaluate(model,id_loader),evaluate(model,ood_loader),evaluate(model,test_loader)
print('RESTORED epoch', st.get('epoch'), 'meta_val', st.get('val_auroc'))
print('EVAL id',round(id_m['auroc'],4),'ood',round(ood_m['auroc'],4),'test',round(test_m['auroc'],4),'ece_test',round(test_m['ece'],4))
assert abs(ood_m['auroc']-float(st['val_auroc']))<0.01, 'Loaded weights are NOT the best epoch — stop and inspect'
(WORK/'camelyon17_resnet18_source_s42.json').write_text(json.dumps(dict(cuda=True,device=torch.cuda.get_device_name(0),
    checkpoint=str(CKPT),best_epoch=st.get('epoch'),id_val=id_m,val_ood=ood_m,test_target=test_m,
    shift=dict(auroc_drop=id_m['auroc']-test_m['auroc'],f1_drop=id_m['f1']-test_m['f1']),history=history),indent=2))

In [ ]:
def bn_params(m):
    for p in m.parameters(): p.requires_grad=False
    ps=[]
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d) and mod.affine:
            mod.weight.requires_grad=True; mod.bias.requires_grad=True; ps += [mod.weight, mod.bias]
    return ps
def tta_mode(m):
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.Dropout): mod.eval()
        elif isinstance(mod, nn.BatchNorm2d): mod.train()
def entropy(z):
    p=torch.sigmoid(z).clamp(1e-6,1-1e-6); return -(p*torch.log(p)+(1-p)*torch.log(1-p)).view(-1)

def adapt(ld, method, lr, e_margin=0.4):
    m=load_fresh(); src=load_fresh(); src.eval(); opt=torch.optim.Adam(bn_params(m), lr=lr)
    n_seen=n_kept=0; ys,pb,pa=[],[],[]
    for x,y in tqdm(ld, desc=f'{method} lr={lr}'):
        x=x.to(device)
        with torch.no_grad(): pb.append(torch.sigmoid(src(x)).view(-1).cpu().numpy())
        tta_mode(m); n_seen += x.size(0)
        if method=='tent':
            opt.zero_grad(set_to_none=True); entropy(m(x)).mean().backward(); opt.step(); n_kept += x.size(0)
        else:
            with torch.no_grad(): keep=entropy(m(x))<e_margin
            k=int(keep.sum())
            if k>0:
                opt.zero_grad(set_to_none=True); entropy(m(x))[keep].mean().backward(); opt.step(); n_kept += k
        with torch.no_grad():
            m.eval(); pa.append(torch.sigmoid(m(x)).view(-1).cpu().numpy())
        ys.append(y.view(-1).numpy())
    y,b,a=np.concatenate(ys),np.concatenate(pb),np.concatenate(pa)
    return dict(method=method,lr=lr,e_margin=e_margin if method=='eata' else None,
                coverage=n_kept/max(n_seen,1), before=metrics_of(y,b), after=metrics_of(y,a), harm=flip_table(y,b,a))

print('--- lr sweep on OOD val (hospital 1) — NOT test ---')
sweep=[]
for lr in (1e-4, 2.5e-4, 1e-3):
    r=adapt(ood_loader,'tent',lr); r['split']='ood_val'; sweep.append(r)
    print('TENT val lr',lr,'AUROC',round(r['after']['auroc'],4),'F1',round(r['after']['f1'],4),'harm',round(r['harm']['harm_rate'],4))
best_lr=max(sweep, key=lambda r: r['after']['auroc'])['lr']
print('SELECTED tent lr', best_lr, 'from OOD val')

print('--- freeze lr, evaluate TEST once ---')
tent=adapt(test_loader,'tent',best_lr); tent['split']='test'
eata=adapt(test_loader,'eata',best_lr, e_margin=0.4); eata['split']='test'
print('TEST source', round(test_m['auroc'],4), test_m['f1'], test_m['ece'])
print('TEST tent  ', round(tent['after']['auroc'],4), tent['after']['f1'], tent['after']['ece'], 'harm', tent['harm']['harm_rate'])
print('TEST eata  ', round(eata['after']['auroc'],4), eata['after']['f1'], eata['after']['ece'], 'harm', eata['harm']['harm_rate'], 'cov', eata['coverage'])
out=dict(cuda=True, device=torch.cuda.get_device_name(0), checkpoint=str(CKPT),
         tent_lr_selected_on='ood_val', tent_lr=best_lr, val_sweep=sweep,
         source_test=test_m, tent_test=tent, eata_test=eata)
(WORK/'camelyon17_resnet18_tent_eata_s42.json').write_text(json.dumps(out, indent=2))
print('download source json, tent_eata json, and', CKPT)